In [62]:
import pandas as pd
import altair as alt
import geopandas as gpd
from shapely import wkt

In [63]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)


neigh_layer = alt.Chart(neigh).mark_geoshape(
    stroke="black",
    fill="lightblue"
).properties(
    width=600,
    height=600
    
).encode(tooltip=['name'])

In [64]:
bus_sol = pd.read_csv('Nodes/N-Bus-Sol.csv')
bus = pd.read_csv('Nodes/N-Bus.csv')
fgc_sol = pd.read_csv('Nodes/N-FGC-Sol.csv')
fgc = pd.read_csv('Nodes/N-FGC.csv')
metro_sol = pd.read_csv('Nodes/N-Metro-Sol.csv')
metro = pd.read_csv('Nodes/N-Metro.csv')
tram_sol = pd.read_csv('Nodes/N-Tram-Sol.csv')
tram = pd.read_csv('Nodes/N-Tram.csv')
all_stops = pd.concat([bus_sol, bus, fgc_sol, fgc, metro_sol, metro, tram_sol, tram], ignore_index=True)


edges_bus = pd.read_csv('Edges/E-Bus.csv')
edges_metro = pd.read_csv('Edges/E-Metro.csv')
edges_fgc = pd.read_csv('Edges/E-FGC.csv')
edges_tram = pd.read_csv('Edges/E-Tram.csv')
all_edges = pd.concat([edges_bus, edges_metro, edges_fgc, edges_tram], ignore_index=True)
exchange = pd.read_csv('Edges/Exchanges_with_Wait_Times.csv')
egress = pd.read_csv('Edges/E-Egress.csv')
pois = pd.read_csv('Nodes/N-POIs.csv')



In [65]:
edges_bus

,origen,dest,tram,linia,type,length,speed,time,directed,cost,geometry
0,B-D20-1284,B-D20-1282,Hospital del Mar - Platja de la Barceloneta,D20,Bus,350.355905,3.333333,1.751780,True,1.751780,LINESTRING (2.194642126631918 41.3831574295074...
1,B-D20-1282,B-D20-1604,Platja de la Barceloneta - Pg Marítim - Pepe R...,D20,Bus,141.475789,3.333333,0.707379,True,0.707379,LINESTRING (2.1928796221773856 41.380294510650...
2,B-D20-1604,B-D20-3348,Pg Marítim - Pepe Rubianes - Pepe Rubianes,D20,Bus,228.766832,3.333333,1.143834,True,1.143834,LINESTRING (2.192167955818947 41.3791384394128...
3,B-D20-3348,B-D20-955,Pepe Rubianes - Pg Joan de Borbó,D20,Bus,353.731837,3.333333,1.768659,True,1.768659,LINESTRING (2.189701539894022 41.3784049721116...
4,B-D20-955,B-D20-1164,Pg Joan de Borbó - Pla de Palau - Pl Pau Vila,D20,Bus,378.388903,3.333333,1.891945,True,1.891945,LINESTRING (2.1873939796176556 41.379986221471...
...,...,...,...,...,...,...,...,...,...,...,...
5408,B-LH1-100754,B-LH1-100757,Mare de Déu de Bellvitge - Trav. Industrial - ...,LH1,Bus,353.353034,3.333333,1.766765,True,1.766765,LINESTRING (2.1045797591984603 41.350548075030...
5409,B-LH1-100757,B-LH1-109282,Poliesportiu Municipal Bellvitge - Campus Univ...,LH1,Bus,167.079814,3.333333,0.835399,True,0.835399,LINESTRING (2.1067845020036153 41.347894321321...
5410,B-LH1-109282,B-LH1-107491,Campus Universitari Bellvitge - Camí Pau Redó ...,LH1,Bus,527.435548,3.333333,2.637178,True,2.637178,"LINESTRING (2.1073692345000006 41.3464769809, ..."
5411,B-LH1-107491,B-LH1-112416,Camí Pau Redó - Institut Català d'Oncologia - ...,LH1,Bus,190.127828,3.333333,0.950639,True,0.950639,"LINESTRING (2.1108209912 41.34318523239999, 2...."


In [66]:
data = pd.read_csv('data/Djisktra-out/export.csv')
origns = data['origin'].unique()
destinations = data['destination'].unique()
all_stops = all_stops[all_stops['id'].isin(list(origns) + list(destinations))]
all_edges = all_edges[all_edges['origen'].isin(origns) & all_edges['dest'].isin(destinations)]
exchanges_solution = data[data['relationship_type'] == 'EXCHANGE']
origin_exchanges = exchanges_solution['origin'].unique()
destination_exchanges = exchanges_solution['destination'].unique()
exchange = exchange[exchange['origen'].isin(origin_exchanges) & exchange['dest'].isin(destination_exchanges)]
egress = egress[egress['origen'].isin(origns) & egress['dest'].isin(destinations)]
pois = pois[pois['poi_name'].isin(destinations)]

In [67]:
all_edges

,origen,dest,tram,linia,type,length,speed,time,directed,cost,geometry
1620,B-7-1070,B-7-149,Gran Via - Pau Claris - Gran Via - Llúria,7,Bus,272.235806,3.333333,1.361179,True,1.361179,LINESTRING (2.1693494217202742 41.390163771096...
1621,B-7-149,B-7-125,Gran Via - Llúria - Metro Tetuan,7,Bus,301.125241,3.333333,1.505626,True,1.505626,LINESTRING (2.1716502846552386 41.391888463562...
1622,B-7-125,B-7-1319,Metro Tetuan - Gran Via - Roger de Flor,7,Bus,390.892073,3.333333,1.954460,True,1.954460,LINESTRING (2.174186962488556 41.3938139348431...
1623,B-7-1319,B-7-914,Gran Via - Roger de Flor - Gran Via - Marina,7,Bus,394.784141,3.333333,1.973921,True,1.973921,LINESTRING (2.1772683123233714 41.396127947786...
1624,B-7-914,B-7-499,Gran Via - Marina - Gran Via - Padilla,7,Bus,299.029820,3.333333,1.495149,True,1.495149,LINESTRING (2.1806121980171262 41.398638785301...
1625,B-7-499,B-7-32,Gran Via - Padilla - Metro Glòries,7,Bus,565.984657,3.333333,2.829923,True,2.829923,LINESTRING (2.1832401858036454 41.400409235106...
1626,B-7-32,B-7-1879,Metro Glòries - Diagonal - Ciutat de Granada,7,Bus,303.452175,3.333333,1.517261,True,1.517261,LINESTRING (2.1880426364190972 41.403051648725...
1627,B-7-1879,B-7-3469,Diagonal - Ciutat de Granada - Tànger - Llacuna,7,Bus,240.827770,3.333333,1.204139,True,1.204139,LINESTRING (2.1912464120929376 41.404224539710...
1628,B-7-3469,B-7-2262,Tànger - Llacuna - Diagonal - Rambla del Poblenou,7,Bus,124.211119,3.333333,0.621056,True,0.621056,LINESTRING (2.1939820540611747 41.404904984710...


In [68]:
all_stops['geometry']= all_stops['geometry'].apply(wkt.loads)
all_stops = gpd.GeoDataFrame(all_stops, geometry='geometry', crs="EPSG:4326")
all_edges['geometry']= all_edges['geometry'].apply(wkt.loads)
all_edges = gpd.GeoDataFrame(all_edges, geometry='geometry', crs="EPSG:4326")
egress['geometry']= egress['geometry'].apply(wkt.loads)
egress = gpd.GeoDataFrame(egress, geometry='geometry', crs="EPSG:4326")
exchange['geometry']= exchange['geometry'].apply(wkt.loads)
exchange = gpd.GeoDataFrame(exchange, geometry='geometry', crs="EPSG:4326")
pois['geometry']= pois['geometry'].apply(wkt.loads)
pois = gpd.GeoDataFrame(pois, geometry='geometry', crs="EPSG:4326")

In [69]:
all_edges[all_edges['origen'].isin(origin_exchanges) & all_edges['dest'].isin(destination_exchanges)]

,origen,dest,tram,linia,type,length,speed,time,directed,cost,geometry


In [70]:
all_edges[all_edges['origen'] == 'B-7-1879']

,origen,dest,tram,linia,type,length,speed,time,directed,cost,geometry
1627,B-7-1879,B-7-3469,Diagonal - Ciutat de Granada - Tànger - Llacuna,7,Bus,240.82777,3.333333,1.204139,True,1.204139,"LINESTRING (2.19125 41.40422, 2.19385 41.40487..."


In [71]:
points_layer = alt.Chart(all_stops).mark_geoshape(size=0).encode(color = 'linia:N',tooltip = ['id:N', 'linia:N'])
edges_layer = alt.Chart(all_edges[all_edges['origen'] == 'B-7-1879']).mark_geoshape(strokeWidth=1,filled=False).encode(color = 'linia:N',tooltip = ['origen:N', 'dest:N', 'linia:N'])
exchanges_layer = alt.Chart(exchange).mark_geoshape(strokeWidth=1,filled=False,color = 'red').encode(tooltip = ['origen:N', 'dest:N', 'linia:N'])
egress_layer = alt.Chart(egress).mark_geoshape(strokeWidth=1,filled=False,color = 'yellow').encode(tooltip = ['origen:N', 'dest:N', 'linia:N'])
poi_layer = alt.Chart(pois).mark_geoshape(stroke="black",color = 'black').encode(tooltip=['poi_name'])
edges_layer + exchanges_layer + points_layer + poi_layer

alt.LayerChart(...)

In [30]:
data

,route_id,step,origin,destination,tram,edge_type,relationship_type,edge_cost,cumulativeCost,totalCost
0,SM-126 to Poblenou / 22@,1,SM-126,B-7-1070,Catalunya - Gran Via - Pau Claris,Exchange,EXCHANGE,17.076524,17.076524,42.076738
1,SM-126 to Poblenou / 22@,2,B-7-1070,B-7-149,Gran Via - Pau Claris - Gran Via - Llúria,Bus,TRAVEL_BUS,1.361179,18.437703,42.076738
2,SM-126 to Poblenou / 22@,3,B-7-149,B-7-125,Gran Via - Llúria - Metro Tetuan,Bus,TRAVEL_BUS,1.505626,19.943329,42.076738
3,SM-126 to Poblenou / 22@,4,B-7-125,B-7-1319,Metro Tetuan - Gran Via - Roger de Flor,Bus,TRAVEL_BUS,1.954460,21.897789,42.076738
4,SM-126 to Poblenou / 22@,5,B-7-1319,B-7-914,Gran Via - Roger de Flor - Gran Via - Marina,Bus,TRAVEL_BUS,1.973921,23.871710,42.076738
5,SM-126 to Poblenou / 22@,6,B-7-914,B-7-499,Gran Via - Marina - Gran Via - Padilla,Bus,TRAVEL_BUS,1.495149,25.366859,42.076738
6,SM-126 to Poblenou / 22@,7,B-7-499,B-7-32,Gran Via - Padilla - Metro Glòries,Bus,TRAVEL_BUS,2.829923,28.196782,42.076738
7,SM-126 to Poblenou / 22@,8,B-7-32,B-7-1879,Metro Glòries - Diagonal - Ciutat de Granada,Bus,TRAVEL_BUS,1.517261,29.714043,42.076738
8,SM-126 to Poblenou / 22@,9,B-7-1879,B-7-3469,Diagonal - Ciutat de Granada - Tànger - Llacuna,Bus,TRAVEL_BUS,1.204139,30.918182,42.076738
9,SM-126 to Poblenou / 22@,10,B-7-3469,B-7-2262,Tànger - Llacuna - Diagonal - Rambla del Poblenou,Bus,TRAVEL_BUS,0.621056,31.539238,42.076738


In [13]:
exchange = pd.read_csv('Edges/Exchanges_with_Wait_Times.csv')
exchange

,origen,dest,tram,mode,lines,type,time,wait_time,directed,cost,geometry
0,ST-FRMC,T-T1-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,IU Stop - T1,Exchange - Self,2.000000,6.601392,True,21.552435,POINT (2.143174886703491 41.3922004699707)
1,T-T2-FRMC,T-T1-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,T2 - T1,Exchange - Self,1.000000,6.601392,True,20.552435,POINT (2.143174886703491 41.3922004699707)
2,T-T3-FRMC,T-T1-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,T3 - T1,Exchange - Self,1.000000,6.601392,True,20.552435,POINT (2.143174886703491 41.3922004699707)
3,ST-FRMC,T-T2-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,IU Stop - T2,Exchange - Self,2.000000,4.218604,True,17.382556,POINT (2.143174886703491 41.3922004699707)
4,T-T1-FRMC,T-T2-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,T1 - T2,Exchange - Self,1.000000,4.218604,True,16.382556,POINT (2.143174886703491 41.3922004699707)
...,...,...,...,...,...,...,...,...,...,...,...
130799,B-M14-109031,M-L5-555,Av. Manuel Azaña - Av. de Xile - Ernest Lluch,Bus - Metro,M14 - L5,Exchange,3.716667,0.943370,True,13.367565,"LINESTRING (2.1096154 41.3775619, 2.1096503 41..."
130800,ST-LLUC,M-L5-555,Ernest Lluch - Ernest Lluch,Tram - Metro,IU Stop - L5,Exchange,1.900000,0.943370,True,11.550898,"LINESTRING (2.11081 41.3765127, 2.1107119 41.3..."
130801,T-T1-LLUC,M-L5-555,Ernest Lluch - Ernest Lluch,Tram - Metro,T1 - L5,Exchange,1.900000,0.943370,True,11.550898,"LINESTRING (2.11081 41.3765127, 2.1107119 41.3..."
130802,T-T2-LLUC,M-L5-555,Ernest Lluch - Ernest Lluch,Tram - Metro,T2 - L5,Exchange,1.900000,0.943370,True,11.550898,"LINESTRING (2.11081 41.3765127, 2.1107119 41.3..."
